In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- PATCH FP16 ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
import math
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Injetar em RoTHP
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except: pass

from easy_tpp.model.torch_model.torch_rothp_decay import RoTHPDecay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")


In [ ]:
class RoTHPDecayForced(RoTHPDecay):
    """
    Versão do RoTHP Decay que força comportamento Hawkes clássico:
    1. Base Intensity removida (ou muito pequena e fixa).
    2. Inicialização de decaimento mais agressiva.
    """
    def __init__(self, model_config):
        super(RoTHPDecayForced, self).__init__(model_config)
        
        # FORÇAR DECAIMENTO MAIS RÁPIDO
        nn.init.uniform_(self.factor_intensity_decay, 3.0, 8.0)
        
        # TRAVAR A BASE EM ZERO
        self.factor_intensity_base.requires_grad = False
        self.factor_intensity_base.data.fill_(0.01) # Base quase nula


In [ ]:
print("Loading Retweet...")
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])
TIME_SCALE = np.mean(all_deltas)
print(f"Time Scale: {TIME_SCALE:.4f}")

NUM_EVENT_TYPES = 3
PAD_TOKEN_ID = NUM_EVENT_TYPES

class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = NUM_EVENT_TYPES
        self.num_event_types_pad = NUM_EVENT_TYPES + 1
        self.pad_token_id = PAD_TOKEN_ID
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), PAD_TOKEN_ID, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / TIME_SCALE
        td = td / TIME_SCALE
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate_fn_opt, num_workers=0)
dev_loader = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate_fn_opt, num_workers=0)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate_fn_opt, num_workers=0)


In [ ]:
def train_model(model_cls, model_name, epochs=30):
    print(f"\n>>> {model_name}")
    config = ModelConfig()
    model = model_cls(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    checkpoint = f'best_decay_{model_name}.pth'
    if os.path.exists(checkpoint):
        print(f"Carregando {model_name}...")
        model.load_state_dict(torch.load(checkpoint))
        return model
        
    best_nll = float('inf')
    for epoch in range(1, epochs+1):
        model.train()
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
        model.eval()
        val_loss = 0
        val_num = 0
        with torch.no_grad():
            for batch in dev_loader:
                batch = [t.to(device) for t in batch]
                with torch.amp.autocast('cuda'):
                    l, n = model.loglike_loss(batch)
                val_loss += l.item()
                val_num += n
        nll = val_loss / (val_num + 1e-9)
        if epoch % 5 == 0:
            print(f"  Ep {epoch}: NLL {nll:.4f}")
        if nll < best_nll:
            best_nll = nll
            torch.save(model.state_dict(), checkpoint)
            
    model.load_state_dict(torch.load(checkpoint))
    return model

models = {}
# Comparar RoTHP Decay Normal vs RoTHP Decay Forçado
for name, cls in [('RoTHP Decay (Normal)', RoTHPDecay), ('RoTHP Decay (Forced)', RoTHPDecayForced)]:
    models[name] = train_model(cls, name, epochs=20)


In [ ]:
def compute_final_metrics(model, loader):
    model.eval()
    total_acc = 0
    total_rmse = 0
    total_nll = 0
    total_events = 0
    
    with torch.no_grad():
        for batch in loader:
            batch_gpu = tuple(t.to(device) for t in batch)
            
            with torch.amp.autocast('cuda'):
                 loss, num_events = model.loglike_loss(batch_gpu)
            total_nll += loss.item()
            
            _, time_delta_target, type_target, mask_target, _ = batch_gpu
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch_gpu)
            
            target_types = type_target[:, 1:]
            target_deltas = time_delta_target[:, 1:]
            target_mask = mask_target[:, 1:]
            
            correct = (types_pred == target_types) * target_mask
            total_acc += correct.sum().item()
            se = ((dtimes_pred - target_deltas) ** 2) * target_mask
            total_rmse += se.sum().item()
            total_events += target_mask.sum().item()
            
    avg_nll = total_nll / (total_events + 1e-9)
    avg_acc = total_acc / (total_events + 1e-9)
    avg_rmse = np.sqrt(total_rmse / (total_events + 1e-9))
    return avg_nll, avg_acc, avg_rmse

print("\n--- Métricas Finais (Teste) ---")
results_list = []
for name, model in models.items():
    nll, acc, rmse = compute_final_metrics(model, test_loader)
    results_list.append({'Model': name, 'NLL': nll, 'Acc': acc, 'RMSE': rmse})
print(pd.DataFrame(results_list))

# Visualização
def visualize_intensity(models, sample_idx=0):
    print(f"\nAnalisando Amostra {sample_idx}...")
    sample = [test_data[sample_idx]]
    batch = collate_fn_opt(sample)
    
    pad_time, pad_delta, pad_type, _, attn = [t.to(device) for t in batch]
    
    t_seq = pad_time[0].cpu().numpy()
    valid_len = (pad_type[0] != PAD_TOKEN_ID).sum().item()
    t_seq = t_seq[:valid_len]
    target_type = pad_type[0, 1].item()
    
    t_start_idx = 1
    t_end_idx = min(5, valid_len-1)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = {'RoTHP Decay (Normal)': 'green', 'RoTHP Decay (Forced)': 'red'}
    
    for i in range(t_start_idx, t_end_idx):
        t_prev = t_seq[i]
        t_next = t_seq[i+1]
        dt_real = t_next - t_prev
        
        dts_scan = torch.linspace(0, dt_real, 100, device=device).view(1, 1, -1)
        
        for name, model in models.items():
            model.eval()
            with torch.no_grad():
                L = pad_time.shape[1]
                sample_dtimes = torch.zeros(1, L, 100, device=device)
                sample_dtimes[:, i, :] = dts_scan.squeeze()
                
                lambdas = model.compute_intensities_at_sample_times(
                    pad_time, pad_delta, pad_type, sample_dtimes, attention_mask=attn
                )
                
                l_curve = lambdas[0, i, :, target_type].cpu().numpy()
                t_curve = t_prev + dts_scan.cpu().numpy().flatten()
                
                ax.plot(t_curve, l_curve, color=colors[name], 
                        label=name if i==t_start_idx else "", linewidth=2)

    for t in t_seq[t_start_idx:t_end_idx+1]:
        ax.axvline(x=t, color='black', alpha=0.2, linestyle=':')
    
    ax.set_title(f'Intensity Function $\\lambda(t)$ - Forced Hawkes Behavior?')
    ax.set_xlabel('Time')
    ax.set_ylabel('Intensity')
    ax.legend()
    plt.show()

    print("\n--- Learned Decay Rates (delta) ---")
    for name, model in models.items():
        decay = F.softplus(model.factor_intensity_decay).detach().cpu().numpy().flatten()
        print(f"{name}: {decay}")

visualize_intensity(models, sample_idx=42)
visualize_intensity(models, sample_idx=100)
